# MNIST手写数字识别实验源码

In [1]:
import paddle
import numpy as np
import matplotlib.pyplot as plt

# 读取MNIST训练集和测试集
paddle.vision.set_image_backend('cv2')
train_dataset = paddle.vision.datasets.MNIST(mode='train')
test_dataset = paddle.vision.datasets.MNIST(mode='test')

# 检查单个样本的输入尺寸和标签
train_data_0 = np.array(train_dataset[0][0])
train_label_0 = np.array(train_dataset[0][1])
print("图像数据形状:", train_data_0.shape)
print("图像标签:", train_label_0)

c:\Users\59718\AppData\Local\Programs\python\Python311\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


图像数据形状: (28, 28)
图像标签: [5]


In [2]:
from paddle.nn import Conv2D, MaxPool2D, Linear
import paddle.nn.functional as F

class MNIST(paddle.nn.Layer):
    def __init__(self):
        super(MNIST, self).__init__()
        self.conv1 = Conv2D(in_channels=1, out_channels=20, kernel_size=5, stride=1, padding=2)
        self.max_pool1 = MaxPool2D(kernel_size=2, stride=2)
        self.conv2 = Conv2D(in_channels=20, out_channels=20, kernel_size=5, stride=1, padding=2)
        self.max_pool2 = MaxPool2D(kernel_size=2, stride=2)
        self.fc = Linear(in_features=980, out_features=10)

    def forward(self, inputs, label=None, training=True):
        x = F.relu(self.conv1(inputs))
        x = self.max_pool1(x)
        x = F.relu(self.conv2(x))
        x = self.max_pool2(x)
        x = paddle.reshape(x, [x.shape[0], -1])
        outputs = self.fc(x)
        if label is not None:
            acc = paddle.metric.accuracy(input=F.softmax(outputs), label=label)
            return outputs, acc
        return outputs

In [3]:
def train(model):
    model.train()
    opt = paddle.optimizer.Adam(learning_rate=0.001, parameters=model.parameters())
    EPOCH_NUM = 10
    for epoch_id in range(EPOCH_NUM):
        for batch_id, data in enumerate(train_loader()):
            images, labels = data
            images = paddle.to_tensor(images)
            labels = paddle.to_tensor(labels)
            predicts, avg_acc = model(images, labels)
            loss = F.cross_entropy(predicts, labels)
            avg_loss = paddle.mean(loss)
            if batch_id % 100 == 0:
                print("epoch: {}, batch: {}, loss is: {}, acc is {}".format(
                    epoch_id, batch_id, avg_loss.numpy(), avg_acc.numpy()))
            avg_loss.backward()
            opt.step()
            opt.clear_grad()
    paddle.save(model.state_dict(), 'mnist.pdparams')